In [3]:
%%writefile mitm_lwe_extremo.cu
#include <iostream>
#include <vector>
#include <chrono>
#include <iomanip>
#include <cuda_runtime.h>
#include <thrust/sort.h>
#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h>

void generar_pesos_oraculo(std::vector<uint64_t>& pesos, int d, uint64_t Q) {
    uint64_t seed = 987654321;
    for (int i = 0; i < d; ++i) {
        seed ^= seed << 21; seed ^= seed >> 35; seed ^= seed << 4;
        pesos[i] = seed % Q;
    }
}

__device__ uint64_t proyectar_residuo(uint64_t vector_id, int dim, const uint64_t* pesos, uint64_t Q) {
    long long suma = 0;
    uint64_t temp = vector_id;
    for (int i = 0; i < dim; ++i) {
        long long coef = (temp % 3) - 1;
        suma = (suma + coef * pesos[i]) % (long long)Q;
        temp /= 3;
    }
    if (suma < 0) suma += Q;
    return static_cast<uint64_t>(suma);
}

__global__ void forward_kernel(uint64_t total_estados, int half_d, const uint64_t* d_pesos, uint64_t Q, uint64_t* d_tabla) {
    uint64_t id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < total_estados) d_tabla[id] = proyectar_residuo(id, half_d, d_pesos, Q);
}

__global__ void backward_kernel(uint64_t total_estados, int half_d, const uint64_t* d_pesos, uint64_t Q, uint64_t target, const uint64_t* __restrict__ d_tabla, unsigned long long* d_supervivientes) {
    uint64_t id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= total_estados) return;

    uint64_t mi_residuo = proyectar_residuo(id, half_d, d_pesos, Q);
    uint64_t objetivo = (target + Q - mi_residuo) % Q;

    uint64_t left = 0;
    uint64_t right = total_estados;
    while (left < right) {
        uint64_t mid = left + (right - left) / 2;
        if (d_tabla[mid] < objetivo) left = mid + 1;
        else right = mid;
    }

    if (left < total_estados && d_tabla[left] == objetivo) {
        unsigned long long colisiones = 0;
        while (left + colisiones < total_estados && d_tabla[left + colisiones] == objetivo) colisiones++;
        atomicAdd(d_supervivientes, colisiones);
    }
}

int main() {
    int d = 36;
    int half_d = d / 2;
    uint64_t total_estados = 387420489ULL; // 3^18 estados
    uint64_t Q = 15000000000000000ULL;     // 1.5 x 10^16

    std::cout << "=================================================================\n";
    std::cout << " MOTOR CUDA MITM EXTREMO PARA RING-LWE (Límite T4)\n";
    std::cout << "=================================================================\n";
    std::cout << "• Dimensión Lógica (d)       : " << d << "\n";
    std::cout << "• Entropía Estructural       : 3^" << d << " (~1.50 x 10^17 vectores)\n";
    std::cout << "• Norma Combinada Ideal (Q)  : " << Q << "\n";

    std::vector<uint64_t> pesos(d);
    generar_pesos_oraculo(pesos, d, Q);
    uint64_t target_secreto = 555555555555555ULL;

    uint64_t* d_pesos;
    cudaMalloc(&d_pesos, d * sizeof(uint64_t));
    cudaMemcpy(d_pesos, pesos.data(), d * sizeof(uint64_t), cudaMemcpyHostToDevice);

    uint64_t* d_tabla;
    size_t bytes = total_estados * sizeof(uint64_t);
    cudaMalloc(&d_tabla, bytes);
    std::cout << "• Memoria VRAM Asignada      : " << std::fixed << std::setprecision(2) << (bytes / 1048576.0) << " MB (Pico ~6.2 GB con Sort)\n";

    unsigned long long* d_supervivientes;
    cudaMalloc(&d_supervivientes, sizeof(unsigned long long));
    cudaMemset(d_supervivientes, 0, sizeof(unsigned long long));

    int blockSize = 256;
    int numBlocks = (total_estados + blockSize - 1) / blockSize;

    auto start = std::chrono::high_resolution_clock::now();
    forward_kernel<<<numBlocks, blockSize>>>(total_estados, half_d, d_pesos, Q, d_tabla);
    cudaDeviceSynchronize();

    thrust::device_ptr<uint64_t> thrust_ptr(d_tabla);
    thrust::sort(thrust::device, thrust_ptr, thrust_ptr + total_estados);
    cudaDeviceSynchronize();
    auto mid = std::chrono::high_resolution_clock::now();

    backward_kernel<<<numBlocks, blockSize>>>(total_estados, half_d, d_pesos + half_d, Q, target_secreto, d_tabla, d_supervivientes);
    cudaDeviceSynchronize();
    auto end = std::chrono::high_resolution_clock::now();

    unsigned long long encontrados = 0;
    cudaMemcpy(&encontrados, d_supervivientes, sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    double law = 150094635296999121.0 / Q; // 3^36 / Q

    std::cout << "-----------------------------------------------------------------\n";
    std::cout << " RESULTADOS DE LA PODA ALGEBRAICA MASIVA\n";
    std::cout << "-----------------------------------------------------------------\n";
    std::cout << "• Supervivientes Teóricos : " << law << "\n";
    std::cout << "• Supervivientes Empíricos: " << encontrados << "\n";
    std::cout << "• Tiempo Fase 1 (Sort HBM): " << std::chrono::duration<double>(mid - start).count() << " s\n";
    std::cout << "• Tiempo Fase 2 (Búsqueda): " << std::chrono::duration<double>(end - mid).count() << " s\n";
    std::cout << "=================================================================\n";

    cudaFree(d_pesos); cudaFree(d_tabla); cudaFree(d_supervivientes);
    return 0;
}

Writing mitm_lwe_extremo.cu


In [4]:
!/usr/local/cuda/bin/nvcc -O3 -arch=sm_75 -use_fast_math mitm_lwe_extremo.cu -o mitm_lwe_extremo
!./mitm_lwe_extremo

 MOTOR CUDA MITM EXTREMO PARA RING-LWE (Límite T4)
• Dimensión Lógica (d)       : 36
• Entropía Estructural       : 3^36 (~1.50 x 10^17 vectores)
• Norma Combinada Ideal (Q)  : 15000000000000000
• Memoria VRAM Asignada      : 2955.78 MB (Pico ~6.2 GB con Sort)
-----------------------------------------------------------------
 RESULTADOS DE LA PODA ALGEBRAICA MASIVA
-----------------------------------------------------------------
• Supervivientes Teóricos : 10.01
• Supervivientes Empíricos: 12
• Tiempo Fase 1 (Sort HBM): 0.51 s
• Tiempo Fase 2 (Búsqueda): 2.67 s
